In [26]:

import sys
import logging
sys.path.append('E:\wangzhilin\QuantumGalaxy')
from QGI.mysql import MYSQL
from datetime import date
import pandas as pd
import numpy as np
import re
def get_logger():
    logger=logging.getLogger('logger')
    logger.setLevel(logging.INFO)
    fh=logging.FileHandler("E:/wangzhilin/QuantumGalaxy/logs/flask_test.log",'a', encoding='utf-8')
    fh.setLevel(logging.DEBUG)
    ch=logging.StreamHandler()
    ch.setLevel(logging.DEBUG)
    formatter=logging.Formatter(fmt='%(asctime)s %(name)-12s %(levelname)-8s %(message)s',datefmt='%m-%d %H:%M')
    ch.setFormatter(formatter)
    fh.setFormatter(formatter)
    logger.addHandler(ch)
    logger.addHandler(fh)
    return logger


In [4]:
logger=get_logger()

In [27]:
def qgcode_generator(mysql:MYSQL)->str:
    '''generate a new code for Exxx or Cxxx  qgcode'''
    #type='e'or 'c'
    sql='select qg_code from qg_indicator_info where qg_code like "%s%%.QG" order by qg_code desc limit 1'%'C'
    #print(sql)
    old=mysql.read_query(sql)
    if len(old)==0:
        num=1
    else:
        num=int(re.findall(r"\d+",old[0][0])[0])+1
    new='C'+str(num).zfill(5)+'.QG'
    
    mysql.close()
    
    return new


In [8]:

loader=QGCodeLoader(issue)
rescode,msg=loader.load()
res='res code : '+str(rescode)+'  '+msg
logger.critical(res)
#jira.add_comment(issue,res)
#issue.update({:})
res

NameError: name 'issue' is not defined

In [5]:
class QGIndicatorUpdater(QGCodeLoader):
    def __init__(self,issue):
        super(QGIndicatorUpdater,self).__init__(issue)
        self.qg_code='C00001.QG'
        self.datadate='2022-07-14'
        self.data=50
        self.result_msg={-1:"issue's status: %s does not allow updating"%self.dict['status'],0:'Success in updating new data for indicator : %s'%self.qg_code}
    def update(self):
        if self.dict['status']!=1:
            return -1,self.result_msg[-1]
        else:
            former=self.mysql_read('select value from customized_data where code like "%s" order by date desc limit 1'%self.qg_code)[0][0]
            chg=(self.data-former)/self.data
            logger.info('change from last data: %f'%chg)
            #issue.update()
            return 0,self.result_msg[0]
            


In [105]:
QGIndicatorUpdater(issue).update()

07-14 16:05 logger       INFO     using username Local_Editor
07-14 16:05 logger       INFO     accessed to database <qgdbs>
07-14 16:05 logger       INFO     got a cursor
07-14 16:05 logger       INFO     Accepted read sql query "select value from customized_data where code like "C00001.QG" order by date desc limit 1"
07-14 16:05 logger       INFO     Closed connection to database <qgdbs>
07-14 16:05 logger       INFO     change from last data: 0.172802


(0, 'Success in updating new data for indicator : C00001.QG')

In [1]:
#from jiraapi import Jira
#jira=Jira()
from jira import JIRA
jira=JIRA('https://research.quantumgalaxy.cn/', basic_auth=('bot2', "jira_bot2"))

In [2]:
issue=jira.issue(26289)
issue.fields.__dict__

{'issuetype': <JIRA IssueType: name='Customized指标', id='10700'>,
 'components': [],
 'timespent': None,
 'timeoriginalestimate': None,
 'description': '这是一个用于测试的自定义指标。',
 'project': <JIRA Project: key='INDICATOR', name='指标跟踪项目', id='10701'>,
 'fixVersions': [],
 'aggregatetimespent': None,
 'timetracking': <JIRA TimeTracking at 2844813550736>,
 'customfield_10104': None,
 'customfield_11711': 'C00005.QG',
 'customfield_10105': '1|i0255r:',
 'customfield_11710': '{"rows":[],"ts":1659333260240}',
 'customfield_10700': None,
 'customfield_10701': None,
 'customfield_10625': None,
 'customfield_10702': None,
 'customfield_10900': None,
 'aggregatetimeestimate': None,
 'customfield_10703': None,
 'customfield_10704': None,
 'resolutiondate': None,
 'workratio': -1,
 'summary': '测试自定义指标',
 'lastViewed': None,
 'watches': <JIRA Watchers at 2844813552800>,
 'creator': <JIRA User: displayName='张凌宇', key='JIRAUSER10000', name='zhly05'>,
 'subtasks': [],
 'created': '2022-07-18T15:35:41.000+0800'

In [3]:
jira.transition_issue(issue,'11')

{}

In [12]:
loader=QGCodeLoader(issue)
loader.dict['status']



<JIRA Status: name='开放', id='1'>

In [22]:
from flask import request,render_template,redirect
import json
from flask import Flask
ALLOWED_IPS=['82.156.232','127.0.0']
app=Flask(__name__)
@app.before_request
def limit_remote_addr():
    
    client_ip = str(request.remote_addr)
    print(client_ip)
    valid = False
    for ip in ALLOWED_IPS:
        if client_ip.startswith(ip) or client_ip == ip:
            valid = True
            logger.info(client_ip)
            break
    if not valid:
        return 'not valid'
@app.route("/head")
def hello_world():
    return "<p>Hello, This is QG's jira2mysql assistant!</p>"


    

@app.route('/add_indicator', methods=['GET','POST'])
def get_body():

    
    payload=request.get_json()
    issueid=payload['issue']['id']
    issue=jira.issue(issueid)
    print(issue)
    #print(type(payload))
    return '200'

def load_new_indicator(test=False):
    if test:
        return request.args
    if request.method=='POST':
        data=request.args
        issueid=data['issueid']
        logger.info('get issue id %s'%issueid)
        issue=jira.issue(issueid)
        logger.info('issue name: %s'%issue.fields.__dict__['summary'])
        res_code,res_msg=QGCodeLoader(issue).load()
        
        res='res code : '+str(res_code)+'  '+res_msg
        jira.add_comment(issue,res)
        logger.critical(res)
        if res_code>=0:
            jira.transition_issue(issue,'11')       
        return json.dumps({'code':0},ensure_ascii=False)
        
    elif request.method=='GET':
        return json.dumps({'msg':'not allowed method!'})

@app.route('/update_indicator')
def update_indicator():
    issueid=request.args['issueid']
    logger.info('get issue id %s'%issueid)
    issue=jira.issue(issueid)
    logger.info('issue name: %s'%issue.fields.__dict__['summary'])
    res_code,res_msg=QGIndicatorUpdater(issue).update()

In [23]:
app.run(host='0.0.0.0',port=81)

 * Serving Flask app '__main__' (lazy loading)
 * Environment: production
   Use a production WSGI server instead.
 * Debug mode: off


 * Running on all addresses.
 * Running on http://172.21.0.14:81/ (Press CTRL+C to quit)
07-22 15:22 logger       INFO     82.156.232.162
82.156.232.162 - - [22/Jul/2022 15:22:36] "POST /add_indicator?issueid=26289&user_id=wangzhilin&user_key=JIRAUSER10821 HTTP/1.1" 200 -


82.156.232.162
INDICATOR-1


In [108]:
issue.update({'customfield_10201':'C00005.QG'})

JIRAError: JiraError HTTP 400 url: https://research.quantumgalaxy.cn/rest/api/2/issue/25810
	text: Field 'customfield_10201' cannot be set. It is not on the appropriate screen, or unknown.
	
	response headers = {'X-AREQUESTID': '722x2222609x7', 'X-ASESSIONID': '1g2ka41', 'Referrer-Policy': 'strict-origin-when-cross-origin', 'X-XSS-Protection': '1; mode=block', 'X-Content-Type-Options': 'nosniff', 'X-Frame-Options': 'SAMEORIGIN', 'Content-Security-Policy': "frame-ancestors 'self'", 'Strict-Transport-Security': 'max-age=31536000', 'X-Seraph-LoginReason': 'OK', 'X-AUSERNAME': 'wangzhilin', 'Cache-Control': 'no-cache, no-store, no-transform', 'Content-Encoding': 'gzip', 'Vary': 'User-Agent', 'Content-Type': 'application/json;charset=UTF-8', 'Transfer-Encoding': 'chunked', 'Date': 'Thu, 14 Jul 2022 04:02:05 GMT', 'Connection': 'close'}
	response text = {"errorMessages":[],"errors":{"customfield_10201":"Field 'customfield_10201' cannot be set. It is not on the appropriate screen, or unknown.

In [145]:
re.findall(r"(\w+.\w+)",'321321.sg')

['321321.sg']

In [24]:
mysql=MYSQL()


NameError: name 'mysql' is not defined

In [32]:
class QGCodeLoader():
    ''' 检查一个输入的qgnode信息是否合法，合法则录入qgindicatorinfo并根据分类录入下属的三个info，并录入其他相关信息；不合法则不入库并返回错误信息。
    '''
    def __init__(self, issue):
        #self.mysql=kwargs.get('mysql',False)

        self.issue = issue
        #self.issue = None
        
        d = issue.fields.__dict__
        self.already_had_code=d.get('customfield_11711',None)
        self.category = d['issuetype'].id  #ticker:10704 #edb:10703 c:10700
        t = d.get('customfield_11009', None)
        if t:
            self.type = t.value  #万得EDB 同花顺EDB
        else:
            self.type = None
        self.source_code = d.get('customfield_11429', None)

        self.name = d['summary']
        self.unit = d.get('customfield_11431', None)
        self.desc = d['description']
        self.status = d['status'].id  #1:开放 ：监测中

        #self.dict['qg_code']='321321.sg'

        self.result_msg = {
            0: 'good indicator',
            -101: 'bad indicator:no TYPE found for ticker indicator',
            -102: 'got wrong type for edb indicator, expect 万得 or 同花顺',
            -103: 'no UNIT found for edb indicator',
            -104: 'got invalid category',
            -201: 'already had an indicator with same qg_code',
            -202: 'got invalid code for some reason',
            -203: 'got invalid indicator name'
        }

    def _get_mysql(self):
        sql_host = 'localhost'
        sql_user = 'Local_Editor'
        sql_password = 'QuantumGalaxy'
        database = 'qgdbs'
        mysql = MYSQL(sql_host, sql_user, sql_password, database)
        return mysql

    def mysql_write(self, sql, val=None):
        mysql = self._get_mysql()
        if val:

            mysql.write_many_query(sql, val)
        else:
            mysql.write_query(sql)
        mysql.close()

    def mysql_read(self, sql):
        mysql = self._get_mysql()
        r = mysql.read_query(sql)
        mysql.close()
        return r

    def check_attr_and_find_target_db(self):
        '''verify nodeinfo is legal ,set default value if not found.
        find target 2nd datebase by category, return 2nd db string'''
        
        if self.category == '10704':
            try:
                self.source_code = re.findall(r"(\w+.\w+)", self.source_code)[0]
            except :
                self.result_msg[-202]='got invalid code:%s' % self.source_code
                return False, -202
            #ticker should have a type like 'stock' or other
            '''
            if not self.dict.get('type', False):

                logger.error('no TYPE found for ticker indicator')
                return False, -101
            if self.dict.get('status', 1) != 0:
                self.dict['status'] = 1
            if not self.dict.get('summary', None):
                #set a default value for args which needed
                self.dict['summary'] = None
            if not self.dict.get('region', None):
                self.dict['region'] = None'''
            self.region = None
            self.ticker_status = 1
            self.qg_code = self.source_code
            self.business = None
            return 'ticker', 0
        elif self.category == '10703':
            try:
                self.source_code = re.findall(r"(\w+\w+)", self.source_code)[0]
            except :
                self.result_msg[-202]='got invalid code:%s' % self.source_code
                return False, -202
            if self.type == '同花顺EDB':
                self.qg_code = self.source_code+'.TH'
            elif self.type == '万得EDB':
                self.qg_code = self.source_code+'.WD'
            else:

                return False, -102

            if not self.unit:
                #logger.error('no UNIT found for edb indicator')
                return False, -103
            '''if not self.dict.get('summary', False):
                self.dict['summary'] = None'''
            #if not self.frequency:
            self.frequency = 0

            return 'edb', 0
        elif self.category == '10700':
            #C类指标重启监测会发一个新的代码，怎么处理
            if self.already_had_code:
                self.qg_code=self.already_had_code
            else:
                self.qg_code = qgcode_generator(self._get_mysql())
            return 'customized', 0
        else:
            #print(self.dict['qg_code'])
            return False, -104

    def check_code(self):
        '''check if the code already in qg_indicator_info'''
        if not self.name:
            self.result_msg[-203] = 'got invalid indicator name: "%s"' % (
                self.name)
            return False, -203

        sql = 'select name from qg_indicator_info where qg_code= "%s"' % self.qg_code
        r = self.mysql_read(sql)
        if r:
            self.result_msg[
                1] = 'Already had a indicator with same qgcode :%s, which name is :%s' % (
                    self.qg_code, r[0][0])
            return True, 1
        else:
            return True, 0

    def load(self):
        #print(self.dict)
        r, result_code = self.check_attr_and_find_target_db()
        if r:
            r1, result_code = self.check_code()
            if result_code==1:
                self.issue.update({'customfield_11711': self.qg_code})
                return 1,self.result_msg[1]
            if result_code==0:
                sql0 = 'insert into qg_indicator_info (qg_code,name,category,description,need_process) value("%s","%s","%s","%s",%s)' % (
                    self.qg_code, self.name, self.category, self.desc,
                    self.status)

                #print(sql0)
                self.mysql_write(sql0)
                if r == 'ticker':
                    sql = 'insert into ticker_info (code,name,region,status,type,business) value ("%s","%s","%s",%s,"%s","%s")' % (
                        self.qg_code, self.name, self.region,
                        self.ticker_status, self.type, self.business)
                    #print(sql)
                    self.mysql_write(sql)
                elif r == 'edb':
                    if self.type == '同花顺EDB':

                        sql = 'insert into edb_info (code,name,ths_code,frequency,unit) value ("%s","%s","%s",%s,"%s") ' % (
                            self.qg_code, self.name, self.source_code,
                            self.frequency, self.unit)
                        #print(sql)
                    else:
                        sql = 'insert into edb_info (code,name,wind_code,frequency,unit) value ("%s","%s","%s",%s,"%s") ' % (
                            self.qg_code, self.name, self.source_code,
                            self.frequency, self.unit)
                        #print(sql)
                    self.mysql_write(sql)
                elif r == 'customized':
                    sql = 'insert into customized_info (code,name) value ("%s","%s")' % (
                        self.qg_code,
                        self.name,
                    )
                    #print(sql)
                    self.mysql_write(sql)
                #issue.update(:self.dict['code'])
                self.issue.update({'customfield_11711': self.qg_code})
                return 0, 'Success in loading indicator: "%s" to qgdbs, whose qg_code is "%s"' % (
                    self.name, self.qg_code)
            else:

                return result_code, self.result_msg[result_code]
        else:

            return result_code, self.result_msg[result_code]

In [33]:
issueid=26414
issue=jira.issue(issueid)
logger.info('issue name: %s'%issue.fields.__dict__['summary'])


07-23 00:35 logger       INFO     issue name: Roblox 每日10pm 前十10游戏在线人数
07-23 00:35 logger       INFO     using username Local_Editor
07-23 00:35 logger       INFO     accessed to database <qgdbs>
07-23 00:35 logger       INFO     got a cursor
07-23 00:35 logger       INFO     Accepted read sql query "select name from qg_indicator_info where qg_code= "C00007.QG""
07-23 00:35 logger       INFO     Closed connection to database <qgdbs>


In [36]:
res_code,res_msg=QGCodeLoader(issue).load()
res_code

07-23 00:39 logger       INFO     using username Local_Editor
07-23 00:39 logger       INFO     accessed to database <qgdbs>
07-23 00:39 logger       INFO     got a cursor
07-23 00:39 logger       INFO     Accepted read sql query "select name from qg_indicator_info where qg_code= "C00007.QG""
07-23 00:39 logger       INFO     Closed connection to database <qgdbs>
07-23 00:39 logger       INFO     using username Local_Editor
07-23 00:39 logger       INFO     accessed to database <qgdbs>
07-23 00:39 logger       INFO     got a cursor
07-23 00:39 logger       WARNING  Had problem on sql "insert into qg_indicator_info (qg_code,name,category,description,need_process) value("C00007.QG","Roblox 每日10pm 前十10游戏在线人数","10700","定义：在美东时间每日10pm，Roblox Popular Worldwide榜单前十游戏在线人数之和的平均值

来源：Roblox App（美区App store)

单位：万人

指示对象：Roblox玩家日活人数

星图坐标：Roblox

任务： 不能连续三天低于120万

异常情况（满足以下任意其一则需向研究员提出警报）：
 1. 未完成任务
 2. 连续三天平均值超160万。",10506)" and rollback
07-23 00:39 logger       WARNING  (1264, "Out of range va

0